# Merge Stage 1 -- Build Panel: Stock Monthly

## Purpose
Combines monthly stock-level cleaned datasets into a single panel keyed on `(permno, date)`, filtered to in-universe stock-months only. This produces the stock-level monthly panel (Panel B).

## Sources (All Cleaned)
- `Data/Data_Collection/Cleaned/07_OpenAssetPricing/oap_monthly_clean.parquet` -- 140 academic factors, month-end dates (the spine)
- `Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_price_targets_clean.parquet` -- 11 factors, month-end dates, keyed on ticker
- `Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_recommendations_clean.parquet` -- 13 factors, month-end dates, keyed on ticker
- `Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_revenue_clean.parquet` -- 12 factors, mid-month dates, keyed on ticker
- `Data/Data_Collection/Cleaned/04_LSEG_IBES/ibes_permno_link_clean.parquet` -- date-aware ticker-to-permno mapping
- `Data/Data_Collection/Cleaned/06_Daily_CRSP_Stock_Data/crsp_daily_clean.parquet` -- month-end price for implied return computation
- `Data/Data_Collection/Cleaned/01_Top100_SP500_Universe/universe_annual_clean.parquet` -- defines which (permno, year) pairs are in-universe

## Key Design Decisions
- **All monthly-to-monthly merges use year-month (`_ym`) keys, not exact dates.** OAP uses calendar month-end, IBES uses last-business-day-of-month, and CRSP uses last-trading-day-of-month. These can differ by 1--3 days. Merging on exact date would silently produce NaN.
- **IBES revenue uses `merge_asof` with `direction='backward'` and `tolerance=45d`.** Revenue dates are mid-month (~15th). For each OAP month-end row, the most recent IBES revenue snapshot on or before that date is matched. January 15 revenue maps to the January 31 OAP row.
- **OAP factors have no look-ahead bias:** per OAP documentation, a factor dated month M is tradeable by end of month M. Publication lag for accounting data is handled internally via Compustat point-in-time.
- **No winsorisation, no z-standardisation** -- deferred to Stage 2.
- **No forward-fill** -- this is stock-level monthly data where NaN is meaningful.

## Merge Logic

### Step 1: Load Universe
The set of valid `(permno, year)` pairs is loaded from `universe_annual_clean.parquet`.

### Step 2: Load and Filter OAP (Spine)
OAP monthly data is loaded and inner-joined with the universe on `(permno, year)`. OAP serves as the spine -- its month-end dates define the panel's date index. Dates verified as calendar month-end.

### Step 3: Build IBES Ticker-to-PERMNO Mapper
The clean IBES link table is loaded. A helper function `ibes_to_permno` converts any IBES DataFrame from ticker-space to permno-space by merging on `ticker` and filtering to rows where `sdates <= date <= edates`. Handles multi-ticker PERMNOs (e.g., corporate name changes) by selecting the link whose date range contains the IBES observation date. Duplicates on `(permno, date)` after the merge are removed.

### Step 4: Load and Convert IBES Price Targets
Price targets are converted from ticker-space to permno-space using the mapper. Merged onto the spine via `_ym` (year-month period) key.

### Step 5: Load and Convert IBES Recommendations
Recommendations are converted from ticker-space to permno-space. Merged onto the spine via `_ym` key.

### Step 6: Load and Convert IBES Revenue
Revenue estimates are converted from ticker-space to permno-space. Because revenue dates are mid-month (IBES statistical period dates, e.g., January 15), this dataset uses `pd.merge_asof` with `direction='backward'` and a 45-day tolerance instead of the `_ym` key merge used by price targets and recommendations.

### Step 7: Get Month-End Prices from CRSP
CRSP daily data is loaded (only `permno`, `date`, `dlyprc`, `dlycap`). The last row per `(permno, month)` is taken using `.tail(1)` (not `.last()`, which could mix values from different dates if any column has NaN on the final trading day). Month-end price and cap are kept for implied return computation and cap-weighting.

### Step 8: Merge Everything onto OAP Spine
Sequential left joins onto the OAP spine:
1. IBES price targets via `_ym` merge
2. IBES recommendations via `_ym` merge
3. IBES revenue via `merge_asof` backward (mid-month to month-end)
4. Month-end prices via `_ym` merge

Each join is verified to produce no row explosion (row count must remain equal to the spine).

### Step 9: Compute Derived Factors
- **`implied_return`** = `ptg_mean / month_end_price - 1`. Both components are known at month-end, so there is no look-ahead bias. This is the most powerful price target signal per the academic literature.

### Step 10: Final Column Inventory
All columns categorised into ID (`permno`, `date`), weight (`month_end_cap`), price helper (`month_end_price`), and factors. Factor count broken down by source: OAP, IBES Price Targets, IBES Recommendations, IBES Revenue, and derived.

### Step 11: Validation
- No duplicate `(permno, date)` rows
- Row count, PERMNO count, date range, unique date count
- Rows per year with approximate stocks-per-month
- Stocks per month distribution (min, mean, max)
- NaN summary by source (average NaN rate for OAP, IBES PT, IBES Rec, IBES Rev, derived)
- Weight column (`month_end_cap`) checked for NaN and zeros
- Date frequency verification (calendar month-end)
- IBES merge health check: confirms each IBES source has at least some matched rows (guards against silent merge failures)

## Output
`Data/Data_Collection/Final/Stage_1/panel_stock_monthly.parquet` -- keyed on `(permno, date)`, containing ID columns, weight column (`month_end_cap`), price helper (`month_end_price`), and all stock-level monthly factors from OAP, IBES (price targets, recommendations, revenue), and derived (implied return)

In [3]:
# %% [markdown]
# # Merge Pipeline — Notebook 02: Build Panel B (Stock Monthly)
#
# Combines monthly stock-level datasets into a single panel keyed on
# (permno, date), filtered to in-universe stock-months only.
#
# Sources:
#   - OAP monthly (spine) — 140 academic factors, month-end dates
#   - IBES price targets — 11 factors, month-end dates, keyed on ticker
#   - IBES recommendations — 13 factors, month-end dates, keyed on ticker
#   - IBES revenue — 12 factors, mid-month dates, keyed on ticker
#   - CRSP (for month-end price) — needed for implied_return computation
#
# IBES files are keyed on ticker, not permno. The clean link table
# (ibes_permno_link_clean.parquet) provides date-aware ticker→permno mapping.
#
# KEY DESIGN DECISIONS:
#   - All monthly-to-monthly merges use year-month (_ym) keys, NOT exact dates.
#     OAP uses calendar month-end, IBES uses last-business-day-of-month, and
#     CRSP uses last-trading-day-of-month. These can differ by 1-3 days.
#     Merging on exact date would silently produce NaN.
#   - IBES revenue uses merge_asof (mid-month dates → month-end spine).
#   - OAP factors have NO look-ahead bias: per OAP documentation, a factor
#     dated month M is tradeable by end of month M. Publication lag for
#     accounting data is handled internally via Compustat point-in-time.
#   - No winsorisation, no z-standardisation — those happen in Step 2.
#   - No forward-fill — this is stock-level monthly data.
#
# Output: Data/Data_Collection/Final/Stage_1/panel_stock_monthly.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import gc

# ── Paths ────────────────────────────────────────────────────────────────────
CLEANED = Path('../../../Data/Data_Collection/Cleaned')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1')
OUT_DIR.mkdir(parents=True, exist_ok=True)

UNIVERSE_ANNUAL = CLEANED / '01_Top100_SP500_Universe' / 'universe_annual_clean.parquet'
OAP_PATH        = CLEANED / '07_OpenAssetPricing' / 'oap_monthly_clean.parquet'
IBES_LINK_PATH  = CLEANED / '04_LSEG_IBES' / 'ibes_permno_link_clean.parquet'
IBES_PT_PATH    = CLEANED / '04_LSEG_IBES' / 'ibes_price_targets_clean.parquet'
IBES_REC_PATH   = CLEANED / '04_LSEG_IBES' / 'ibes_recommendations_clean.parquet'
IBES_REV_PATH   = CLEANED / '04_LSEG_IBES' / 'ibes_revenue_clean.parquet'
CRSP_PATH       = CLEANED / '06_Daily_CRSP_Stock_Data' / 'crsp_daily_clean.parquet'

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD UNIVERSE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD UNIVERSE")
print("=" * 90)

universe = pd.read_parquet(UNIVERSE_ANNUAL)
print(f"\n  Universe: {len(universe):,} (permno, year) pairs")
print(f"  Unique PERMNOs: {universe['permno'].nunique()}")
print(f"  Year range: {universe['year'].min()} – {universe['year'].max()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: LOAD AND FILTER OAP (SPINE)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: LOAD AND FILTER OAP (SPINE)")
print("=" * 90)

oap = pd.read_parquet(OAP_PATH)
oap['date'] = pd.to_datetime(oap['date'])
print(f"\n  Raw OAP: {len(oap):,} rows × {oap.shape[1]} columns")
print(f"  PERMNOs: {oap['permno'].nunique()}")

# Drop year column if present (from partitioned parquet)
oap = oap.drop(columns=['year'], errors='ignore')

# Filter to universe
oap['_year'] = oap['date'].dt.year
n_before = len(oap)
oap = oap.merge(
    universe[['permno', 'year']],
    left_on=['permno', '_year'],
    right_on=['permno', 'year'],
    how='inner'
).drop(columns=['year', '_year']).reset_index(drop=True)

print(f"  Filtered to universe: {n_before:,} → {len(oap):,} rows")
print(f"  PERMNOs: {oap['permno'].nunique()}")
print(f"  Date range: {oap['date'].min().date()} → {oap['date'].max().date()}")

oap_factor_cols = [c for c in oap.columns if c not in ['permno', 'date']]
print(f"  OAP factors: {len(oap_factor_cols)}")

# Verify dates are month-end
is_month_end = oap['date'].dt.is_month_end
print(f"  Dates that are calendar month-end: {is_month_end.sum():,} / {len(oap):,} "
      f"({is_month_end.mean()*100:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: BUILD IBES TICKER→PERMNO MAPPER
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: BUILD IBES TICKER→PERMNO MAPPER")
print("=" * 90)

link = pd.read_parquet(IBES_LINK_PATH)
link['sdates'] = pd.to_datetime(link['sdates'])
link['edates'] = pd.to_datetime(link['edates'])
print(f"\n  Link table: {len(link)} rows, {link['ticker'].nunique()} tickers, "
      f"{link['permno'].nunique()} PERMNOs")

def ibes_to_permno(ibes_df, link_df, name=''):
    """Convert an IBES DataFrame from ticker-space to permno-space.
    
    For each IBES row, finds the matching link entry where:
      link.ticker == ibes.ticker AND link.sdates <= ibes.date <= link.edates
    
    Handles multi-ticker PERMNOs (e.g., X→MRO1) by picking the link
    whose date range contains the IBES observation date.
    """
    n_before = len(ibes_df)
    tickers_before = ibes_df['ticker'].nunique()
    
    # Merge on ticker (may create multiple matches for multi-PERMNO tickers)
    merged = ibes_df.merge(
        link_df[['ticker', 'permno', 'sdates', 'edates']],
        on='ticker',
        how='inner'
    )
    
    # Filter to valid date range
    merged = merged[
        (merged['date'] >= merged['sdates']) & (merged['date'] <= merged['edates'])
    ].copy()
    
    # Drop link columns
    merged = merged.drop(columns=['ticker', 'sdates', 'edates'])
    
    # Sort before dedup for deterministic behaviour, then deduplicate
    n_dupes = merged.duplicated(subset=['permno', 'date']).sum()
    if n_dupes > 0:
        merged = merged.sort_values(['permno', 'date']).drop_duplicates(
            subset=['permno', 'date'], keep='first'
        )
    
    n_after = len(merged)
    permnos_after = merged['permno'].nunique()
    print(f"  {name}: {n_before:,} rows ({tickers_before} tickers) → "
          f"{n_after:,} rows ({permnos_after} PERMNOs)"
          f"{f', {n_dupes} dupes removed' if n_dupes > 0 else ''}")
    
    return merged

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: LOAD AND CONVERT IBES PRICE TARGETS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: LOAD AND CONVERT IBES PRICE TARGETS")
print("=" * 90)

ibes_pt = pd.read_parquet(IBES_PT_PATH)
ibes_pt['date'] = pd.to_datetime(ibes_pt['date'])
print(f"\n  Raw: {len(ibes_pt):,} rows, {ibes_pt['ticker'].nunique()} tickers")

# Convert to permno-space
ibes_pt = ibes_to_permno(ibes_pt, link, name='Price Targets')

pt_factor_cols = [c for c in ibes_pt.columns if c not in ['permno', 'date']]
print(f"  Factors: {len(pt_factor_cols)} — {pt_factor_cols}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: LOAD AND CONVERT IBES RECOMMENDATIONS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: LOAD AND CONVERT IBES RECOMMENDATIONS")
print("=" * 90)

ibes_rec = pd.read_parquet(IBES_REC_PATH)
ibes_rec['date'] = pd.to_datetime(ibes_rec['date'])
print(f"\n  Raw: {len(ibes_rec):,} rows, {ibes_rec['ticker'].nunique()} tickers")

ibes_rec = ibes_to_permno(ibes_rec, link, name='Recommendations')

rec_factor_cols = [c for c in ibes_rec.columns if c not in ['permno', 'date']]
print(f"  Factors: {len(rec_factor_cols)} — {rec_factor_cols}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: LOAD AND CONVERT IBES REVENUE (MID-MONTH DATES)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: LOAD AND CONVERT IBES REVENUE (MID-MONTH DATES)")
print("=" * 90)

ibes_rev = pd.read_parquet(IBES_REV_PATH)
ibes_rev['date'] = pd.to_datetime(ibes_rev['date'])
print(f"\n  Raw: {len(ibes_rev):,} rows, {ibes_rev['ticker'].nunique()} tickers")
print(f"  Date example (mid-month): {ibes_rev['date'].iloc[0].date()}")

ibes_rev = ibes_to_permno(ibes_rev, link, name='Revenue')

rev_factor_cols = [c for c in ibes_rev.columns if c not in ['permno', 'date']]
print(f"  Factors: {len(rev_factor_cols)} — {rev_factor_cols}")

# Sort for merge_asof (required)
ibes_rev = ibes_rev.sort_values(['permno', 'date']).reset_index(drop=True)

del link
gc.collect()

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: GET MONTH-END PRICES FROM CRSP
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 7: GET MONTH-END PRICES FROM CRSP")
print("=" * 90)

crsp = pd.read_parquet(CRSP_PATH, columns=['permno', 'date', 'dlyprc', 'dlycap'])
crsp['date'] = pd.to_datetime(crsp['date'])
crsp['_ym'] = crsp['date'].dt.to_period('M')

# Take the LAST ROW per (permno, month) — the last trading day of each month.
# Using .tail(1) instead of .last() to ensure all columns come from the same row.
# .last() returns the last non-null value per column independently, which could
# mix values from different dates if any column has NaN on the final trading day.
month_end = (
    crsp.sort_values('date')
    .groupby(['permno', '_ym'])
    .tail(1)
    .reset_index(drop=True)
)

# Keep _ym for merging — do NOT convert to calendar month-end date.
# OAP uses calendar month-end (Jan 31), CRSP's last trading day might be Jan 29.
# Merging on _ym (year-month period) avoids this 1-3 day mismatch.
month_end = month_end[['permno', '_ym', 'dlyprc', 'dlycap']].rename(
    columns={'dlyprc': 'month_end_price', 'dlycap': 'month_end_cap'}
)

del crsp
gc.collect()

print(f"\n  Month-end prices: {len(month_end):,} rows")
print(f"  PERMNOs: {month_end['permno'].nunique()}")
print(f"  Price NaN: {month_end['month_end_price'].isna().sum()}")
print(f"  Cap NaN: {month_end['month_end_cap'].isna().sum()}")
print(f"  Price range: [{month_end['month_end_price'].min():.2f}, "
      f"{month_end['month_end_price'].max():.2f}]")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: MERGE EVERYTHING ONTO OAP SPINE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 8: MERGE EVERYTHING ONTO OAP SPINE")
print("=" * 90)

panel = oap.copy()
del oap
n_spine = len(panel)
print(f"\n  OAP spine: {n_spine:,} rows")

# Add year-month key to spine for _ym merges
panel['_ym'] = panel['date'].dt.to_period('M')

# ── 8a. Join IBES price targets (month-end → _ym merge) ─────────────────────
# IBES PT dates may not exactly match OAP calendar month-end dates,
# so merge on (permno, year-month) instead of (permno, date).
ibes_pt['_ym'] = ibes_pt['date'].dt.to_period('M')
ibes_pt = ibes_pt.drop(columns='date')  # drop IBES date, keep OAP date

n_before = len(panel)
panel = panel.merge(ibes_pt, on=['permno', '_ym'], how='left')
del ibes_pt
assert len(panel) == n_before, f"Row explosion after PT join: {n_before} → {len(panel)}"

n_matched = panel[pt_factor_cols[0]].notna().sum()
print(f"  + Price Targets (_ym): {n_matched:,} / {n_spine:,} matched "
      f"({n_matched/n_spine*100:.1f}%)")

# ── 8b. Join IBES recommendations (month-end → _ym merge) ───────────────────
ibes_rec['_ym'] = ibes_rec['date'].dt.to_period('M')
ibes_rec = ibes_rec.drop(columns='date')

n_before = len(panel)
panel = panel.merge(ibes_rec, on=['permno', '_ym'], how='left')
del ibes_rec
assert len(panel) == n_before, f"Row explosion after REC join: {n_before} → {len(panel)}"

n_matched = panel[rec_factor_cols[0]].notna().sum()
print(f"  + Recommendations (_ym): {n_matched:,} / {n_spine:,} matched "
      f"({n_matched/n_spine*100:.1f}%)")

# ── 8c. Join IBES revenue (mid-month dates → merge_asof backward) ───────────
# Revenue dates are mid-month (~15th). For each OAP (permno, month-end date),
# find the most recent IBES revenue snapshot on or before that date.
# January 15 revenue → maps to January 31 OAP row.
panel = panel.sort_values('date').reset_index(drop=True)
ibes_rev = ibes_rev.sort_values('date').reset_index(drop=True)

n_before = len(panel)
ibes_rev['permno'] = ibes_rev['permno'].astype('int64')
panel = pd.merge_asof(
    panel,
    ibes_rev,
    on='date',
    by='permno',
    direction='backward',
    tolerance=pd.Timedelta('45d')  # max 45 days back (within same month + buffer)
)
del ibes_rev
assert len(panel) == n_before, f"Row explosion after REV join: {n_before} → {len(panel)}"

n_matched = panel[rev_factor_cols[0]].notna().sum()
print(f"  + Revenue (asof): {n_matched:,} / {n_spine:,} matched "
      f"({n_matched/n_spine*100:.1f}%)")

# ── 8d. Join month-end prices (via _ym merge) ───────────────────────────────
n_before = len(panel)
panel = panel.merge(month_end, on=['permno', '_ym'], how='left')
del month_end
assert len(panel) == n_before, f"Row explosion after price join: {n_before} → {len(panel)}"

n_matched = panel['month_end_price'].notna().sum()
print(f"  + Month-end price (_ym): {n_matched:,} / {n_spine:,} matched "
      f"({n_matched/n_spine*100:.1f}%)")

# Drop the _ym helper column
panel = panel.drop(columns='_ym')

gc.collect()

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9: COMPUTE DERIVED FACTORS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 9: COMPUTE DERIVED FACTORS")
print("=" * 90)

# Implied return = consensus price target / current price - 1
# Both components are known at month-end — no look-ahead bias.
if all(c in panel.columns for c in ['ptg_mean', 'month_end_price']):
    panel['implied_return'] = (
        panel['ptg_mean'] / panel['month_end_price'].replace(0, np.nan) - 1
    )
    valid = panel['implied_return'].notna().sum()
    print(f"\n  implied_return: {valid:,} valid values ({valid/len(panel)*100:.1f}%)")
    print(f"    Median: {panel['implied_return'].median():.4f}")
    print(f"    Mean:   {panel['implied_return'].mean():.4f}")
    print(f"    Range:  [{panel['implied_return'].min():.4f}, "
          f"{panel['implied_return'].max():.4f}]")
    
    # Sanity: implied return should mostly be positive (analysts are optimistic)
    pct_positive = (panel['implied_return'] > 0).mean() * 100
    print(f"    % positive (analyst optimism): {pct_positive:.1f}%")
else:
    print(f"\n  ⚠ Cannot compute implied_return — missing ptg_mean or month_end_price")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 10: FINAL COLUMN INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 10: FINAL COLUMN INVENTORY")
print("=" * 90)

id_cols = ['permno', 'date']
weight_cols = ['month_end_cap']
price_cols = ['month_end_price']  # helper for implied_return, not a factor itself
all_factor_cols = [c for c in panel.columns
                   if c not in id_cols + weight_cols + price_cols]

# Breakdown by source
oap_in_panel = [c for c in oap_factor_cols if c in panel.columns]
pt_in_panel = [c for c in pt_factor_cols if c in panel.columns]
rec_in_panel = [c for c in rec_factor_cols if c in panel.columns]
rev_in_panel = [c for c in rev_factor_cols if c in panel.columns]
derived = ['implied_return'] if 'implied_return' in panel.columns else []
other = [c for c in all_factor_cols
         if c not in oap_in_panel + pt_in_panel + rec_in_panel + rev_in_panel + derived]

print(f"\n  Factor breakdown by source:")
print(f"    OAP:                  {len(oap_in_panel):>4d} factors")
print(f"    IBES Price Targets:   {len(pt_in_panel):>4d} factors")
print(f"    IBES Recommendations: {len(rec_in_panel):>4d} factors")
print(f"    IBES Revenue:         {len(rev_in_panel):>4d} factors")
print(f"    Derived:              {len(derived):>4d} factors")
if other:
    print(f"    Other:                {len(other):>4d} — {other}")
print(f"    ────────────────────────")
print(f"    Total:                {len(all_factor_cols):>4d} factors")
print(f"\n  Weight: month_end_cap (for cap-weighting in aggregation)")
print(f"  Price helper: month_end_price (used to compute implied_return)")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 11: VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 11: VALIDATION")
print("=" * 90)

# ── 11a. No duplicates ──────────────────────────────────────────────────────
n_dupes = panel.duplicated(subset=['permno', 'date']).sum()
print(f"\n  Duplicate (permno, date): {n_dupes}")
assert n_dupes == 0, f"Found {n_dupes} duplicates!"

# ── 11b. Shape ──────────────────────────────────────────────────────────────
print(f"\n  Total rows: {len(panel):,}")
print(f"  Total columns: {panel.shape[1]}")
print(f"  PERMNOs: {panel['permno'].nunique()}")
print(f"  Date range: {panel['date'].min().date()} → {panel['date'].max().date()}")
print(f"  Unique dates: {panel['date'].nunique():,}")

# ── 11c. Rows per year ──────────────────────────────────────────────────────
print(f"\n  Rows per year:")
rows_year = panel.groupby(panel['date'].dt.year).size()
for y, n in rows_year.items():
    avg = n / 12
    print(f"    {y}: {n:>5,d} rows (~{avg:.0f} stocks/month)")

# ── 11d. Stocks per month ───────────────────────────────────────────────────
stocks_per_month = panel.groupby('date')['permno'].nunique()
print(f"\n  Stocks per month:")
print(f"    Mean: {stocks_per_month.mean():.1f}")
print(f"    Min:  {stocks_per_month.min()} (on {stocks_per_month.idxmin().date()})")
print(f"    Max:  {stocks_per_month.max()} (on {stocks_per_month.idxmax().date()})")

# ── 11e. NaN summary by source ──────────────────────────────────────────────
print(f"\n  NaN summary by source:")
for label, cols in [('OAP', oap_in_panel),
                     ('IBES PT', pt_in_panel),
                     ('IBES Rec', rec_in_panel),
                     ('IBES Rev', rev_in_panel),
                     ('Derived', derived)]:
    present_cols = [c for c in cols if c in panel.columns]
    if not present_cols:
        continue
    nan_rate = panel[present_cols].isna().mean().mean() * 100
    n_nan = panel[present_cols].isna().sum().sum()
    total = len(panel) * len(present_cols)
    print(f"    {label:<18s} {nan_rate:>5.2f}% avg NaN  "
          f"({n_nan:,} / {total:,} cells)")

# ── 11f. Weight column ──────────────────────────────────────────────────────
print(f"\n  Weight column (month_end_cap):")
print(f"    NaN: {panel['month_end_cap'].isna().sum()}")
print(f"    Zero: {(panel['month_end_cap'] == 0).sum()}")
if panel['month_end_cap'].notna().any():
    print(f"    Range: [{panel['month_end_cap'].min():,.0f}, "
          f"{panel['month_end_cap'].max():,.0f}]")

# ── 11g. Date frequency check ──────────────────────────────────────────────
is_month_end = panel['date'].dt.is_month_end
print(f"\n  Dates that are calendar month-end: {is_month_end.sum():,} "
      f"({is_month_end.mean()*100:.1f}%)")

# ── 11h. IBES merge health check ───────────────────────────────────────────
# If _ym merge failed silently, ALL IBES cols would be NaN for every row.
# Check that at least some rows have IBES data.
print(f"\n  IBES merge health check:")
for label, cols in [('PT', pt_in_panel), ('Rec', rec_in_panel), ('Rev', rev_in_panel)]:
    present_cols = [c for c in cols if c in panel.columns]
    if present_cols:
        all_nan_rows = panel[present_cols].isna().all(axis=1).sum()
        any_valid = panel[present_cols].notna().any(axis=1).sum()
        print(f"    {label}: {any_valid:,} rows with data, "
              f"{all_nan_rows:,} all-NaN ({all_nan_rows/len(panel)*100:.1f}%)")

# ── 11i. Sample ─────────────────────────────────────────────────────────────
print(f"\n  Sample (first 5 rows, selected columns):")
sample_cols = ['permno', 'date', 'month_end_price', 'month_end_cap',
               'ptg_mean', 'implied_return', 'rec_mean', 'rev_mean_fy1']
sample_cols = [c for c in sample_cols if c in panel.columns]
print(panel[sample_cols].head(5).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 12: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 12: SAVE")
print("=" * 90)

panel = panel.sort_values(['permno', 'date']).reset_index(drop=True)

out_path = OUT_DIR / 'panel_stock_monthly.parquet'
panel.to_parquet(out_path, index=False, engine='pyarrow')

print(f"\n  ✓ Saved: {out_path}")
print(f"    {len(panel):,} rows × {panel.shape[1]} columns")
print(f"    ID: permno, date")
print(f"    Weight: month_end_cap")
print(f"    Price: month_end_price (for implied_return)")
print(f"    Factors: {len(all_factor_cols)}")
print(f"    Size: {out_path.stat().st_size / 1e6:.1f} MB")

print("\nPanel B (stock monthly) complete.")

STEP 1: LOAD UNIVERSE

  Universe: 2,100 (permno, year) pairs
  Unique PERMNOs: 227
  Year range: 2004 – 2024

STEP 2: LOAD AND FILTER OAP (SPINE)

  Raw OAP: 48,861 rows × 143 columns
  PERMNOs: 227
  Filtered to universe: 48,861 → 25,194 rows
  PERMNOs: 227
  Date range: 2004-01-31 → 2024-12-31
  OAP factors: 140
  Dates that are calendar month-end: 25,194 / 25,194 (100.0%)

STEP 3: BUILD IBES TICKER→PERMNO MAPPER

  Link table: 232 rows, 231 tickers, 227 PERMNOs

STEP 4: LOAD AND CONVERT IBES PRICE TARGETS

  Raw: 49,310 rows, 230 tickers
  Price Targets: 49,310 rows (230 tickers) → 48,808 rows (227 PERMNOs), 3 dupes removed
  Factors: 11 — ['ptg_mean', 'ptg_median', 'ptg_high', 'ptg_low', 'ptg_numest', 'ptg_dispersion', 'ptg_range', 'ptg_upside_skew', 'ptg_revision', 'ptg_revision_3m', 'ptg_numest_chg']

STEP 5: LOAD AND CONVERT IBES RECOMMENDATIONS

  Raw: 49,110 rows, 230 tickers
  Recommendations: 49,110 rows (230 tickers) → 48,609 rows (227 PERMNOs), 3 dupes removed
  Factors: 